# Bipartite Cholesky Graph Networks

Implementation of factorized message passing over density-fitted ERI tensors. Generates the benchmark dataset, trains the GNN, and outputs PES and LOMO evaluations.

In [ ]:
# Cell 1 — Setup and Imports
import os, copy, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pyscf import gto, scf, df, fci
from sklearn.model_selection import KFold
import matplotlib.pyplot as plt

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Hardware
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Compute target: {device}")

# Directories
os.makedirs("data_tensors", exist_ok=True)
os.makedirs("figures",      exist_ok=True)
print("data_tensors/ and figures/ ready.")

In [ ]:
# Cell 2 — Cholesky Integral Extraction
#
def compute_cholesky_integrals(mol_str, basis="sto-3g", threshold=1e-6, spin=0):
    """
    Extract one-body integrals (h_core) and density-fitted Cholesky vectors (B).

    Parameters
    ----------
    mol_str   : PySCF atom string, e.g. 'O 0 0 0; O 0 0 1.2'
    basis     : basis set name
    threshold : sparsification threshold
    spin      : unpaired electrons (0=RHF, 2=ROHF triplet, ...)

    Returns
    -------
    h_core     : (N, N)
    B_cholesky : (N_aux_active, N, N)
    e_scf      : SCF total energy (Ha)
    """
    mol = gto.M(atom=mol_str, basis=basis, spin=spin, verbose=0)
    mf  = scf.RHF(mol).run() if spin == 0 else scf.ROHF(mol).run()

    N      = mol.nao_nr()
    h_core = mf.get_hcore()

    mydf = df.DF(mol)
    mydf.auxbasis = df.make_auxbasis(mol, mp2fit=True)
    mydf.build()
    B_raw = mydf._cderi          # (N_aux, N*(N+1)/2) packed triangular

    N_aux      = B_raw.shape[0]
    B_cholesky = np.zeros((N_aux, N, N))

    ij = 0
    for i in range(N):
        for j in range(i + 1):
            B_cholesky[:, i, j] = B_raw[:, ij]
            B_cholesky[:, j, i] = B_raw[:, ij]
            ij += 1

    mask       = np.max(np.abs(B_cholesky), axis=(1, 2)) > threshold
    B_cholesky = B_cholesky[mask]

    return h_core, B_cholesky, mf.e_tot


# Sanity check
_h, _B, _E = compute_cholesky_integrals("H 0 0 0; H 0 0 0.74")
print(f"H2 check: N_orb={_h.shape[0]}, N_aux(active)={_B.shape[0]}, E_HF={_E:.4f} Ha")

In [ ]:
# Cell 3 — Dataset Generation
#
molecules_config = {
    "CO":  {"atoms": ["C",  "O"],  "distances": np.linspace(0.5, 2.5, 22), "spin": 0},
    "HF":  {"atoms": ["H",  "F"],  "distances": np.linspace(0.5, 2.5, 22), "spin": 0},
    "Li2": {"atoms": ["Li", "Li"], "distances": np.linspace(1.5, 3.5, 22), "spin": 0},
    "LiH": {"atoms": ["Li", "H"],  "distances": np.linspace(0.5, 3.0, 22), "spin": 0},
    "N2":  {"atoms": ["N",  "N"],  "distances": np.linspace(0.5, 2.5, 22), "spin": 0},
    "O2":  {"atoms": ["O",  "O"],  "distances": np.linspace(0.5, 2.5, 22), "spin": 2},
}

data_records = []
print("Generating dataset (this may take several minutes on first run).\n")

for mol_name, cfg in molecules_config.items():
    a1, a2 = cfg["atoms"]
    spin   = cfg["spin"]
    print(f"  {mol_name}  spin={spin}  ({len(cfg['distances'])} geometries)")

    for geom_id, r in enumerate(cfg["distances"]):
        mol_str = f"{a1} 0 0 0; {a2} 0 0 {r:.4f}"
        try:
            mol = gto.M(atom=mol_str, basis="sto-3g", spin=spin, verbose=0)
            mf  = scf.RHF(mol).run() if spin == 0 else scf.ROHF(mol).run()

            cisolver = fci.FCI(mf)
            e_fci, _ = cisolver.kernel()

            h, B, e_hf = compute_cholesky_integrals(
                mol_str, basis="sto-3g", threshold=1e-6, spin=spin
            )

            np.save(f"data_tensors/{mol_name}_{geom_id}_h.npy", h)
            np.save(f"data_tensors/{mol_name}_{geom_id}_B.npy", B)

            data_records.append({
                "molecule": mol_name, "geom_id": geom_id,
                "r": r, "spin": spin, "E_HF": e_hf, "E_FCI": e_fci,
            })

        except Exception as exc:
            print(f"    FAILED {mol_name} r={r:.2f}: {exc}")

df_main = pd.DataFrame(data_records)
df_main.to_csv("data_tensors/dataset_index.csv", index=False)
print(f"\nDataset ready: {len(df_main)} geometries across "
      f"{df_main['molecule'].nunique()} molecules.")
print(df_main.groupby("molecule").size().rename("n_geom").reset_index().to_string(index=False))

In [ ]:
# Cell 4 — Model: Factorized Bipartite GNN
#
# Message passing follows the tensor contraction paths of the Cholesky metric
# (consistent with Appendix A.3, Equations 5-8):
#
#   Orbital->Auxiliary:  m_L[h] = sum_{p,q} B^L_pq * x_p[h] * x_q[h]
#   Auxiliary->Orbital:  m_p[h] = sum_{L,q} B^L_pq * h_L[h] * x_q[h]

class FactorizedBipartiteGNN(nn.Module):

    def __init__(self, node_dim=2, hidden_dim=64, num_layers=3):
        super().__init__()
        self.num_layers = num_layers
        self.node_embed = nn.Linear(node_dim, hidden_dim)

        self.W_orb_to_aux = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.LayerNorm(hidden_dim),
                nn.LeakyReLU(),
            ) for _ in range(num_layers)
        ])
        self.W_aux_to_orb = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.LayerNorm(hidden_dim),
                nn.LeakyReLU(),
            ) for _ in range(num_layers)
        ])
        self.energy_head = nn.Sequential(
            nn.Linear(hidden_dim, 128), nn.LeakyReLU(), nn.Dropout(0.1),
            nn.Linear(128, 64),        nn.LeakyReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, h_diag, B_chol):
        # h_diag : (B, N,     2)
        # B_chol : (B, N_aux, N, N)
        B_sz, N_aux, N, _ = B_chol.shape
        x_orb = self.node_embed(h_diag)
        x_aux = torch.zeros(B_sz, N_aux, x_orb.shape[-1], device=x_orb.device)

        for layer in range(self.num_layers):
            # Orbital -> Auxiliary (O(N) memory scaling per channel)
            ket_contract = torch.einsum("blpq,bqh->blph", B_chol, x_orb)
            m_aux        = torch.einsum("blph,bph->blh", ket_contract, x_orb)
            x_aux        = x_aux + self.W_orb_to_aux[layer](m_aux)

            # Auxiliary -> Orbital
            aux_bc = torch.einsum("blh,blpq->blpqh", x_aux, B_chol)
            m_orb  = torch.einsum("blpqh,bqh->bph", aux_bc, x_orb)
            x_orb  = x_orb + self.W_aux_to_orb[layer](m_orb)

        return self.energy_head(x_orb.sum(dim=1)).squeeze(-1)


# Forward-pass sanity check
_m = FactorizedBipartiteGNN().to(device)
print(f"Forward pass shape: {_m(torch.randn(4,6,2,device=device), torch.randn(4,12,6,6,device=device)).shape}")
del _m

In [ ]:
# Cell 5 — Dataset and Collate
#
# Target: delta-ML correlation energy  dE = E_FCI - E_HF
# This removes the large (~10^2 Ha) mean-field contribution and isolates
# the many-body signal (~10^-1 Ha), which stabilises the loss landscape.
# See Appendix B for justification.

class MoleculeDataset(Dataset):

    def __init__(self, data_df, data_dir="data_tensors"):
        self.data     = data_df.reset_index(drop=True)
        self.data_dir = data_dir

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        h   = np.load(f"{self.data_dir}/{row['molecule']}_{int(row['geom_id'])}_h.npy")
        B   = np.load(f"{self.data_dir}/{row['molecule']}_{int(row['geom_id'])}_B.npy")

        # Node features: diagonal orbital energy + row norm of h_core
        node_feats = np.stack([h.diagonal(), np.linalg.norm(h, axis=1)], axis=-1)
        e_corr     = float(row["E_FCI"] - row["E_HF"])

        return {
            "h_feat": torch.FloatTensor(node_feats),
            "B_chol": torch.FloatTensor(B),
            "energy": torch.FloatTensor([e_corr]),
        }


def pad_collate(batch):
    max_N = max(item["h_feat"].shape[0] for item in batch)
    max_L = max(item["B_chol"].shape[0] for item in batch)
    h_out, B_out, E_out = [], [], []
    for item in batch:
        N, L = item["h_feat"].shape[0], item["B_chol"].shape[0]
        h_pad = torch.zeros(max_N, 2);         h_pad[:N]       = item["h_feat"]
        B_pad = torch.zeros(max_L, max_N, max_N); B_pad[:L,:N,:N] = item["B_chol"]
        h_out.append(h_pad); B_out.append(B_pad); E_out.append(item["energy"])
    return {"h_feat": torch.stack(h_out), "B_chol": torch.stack(B_out),
            "energy": torch.cat(E_out)}

In [ ]:
# Cell 6 — Training Engine

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total = 0.0
    for batch in loader:
        h, B, tgt = (batch["h_feat"].to(device),
                     batch["B_chol"].to(device),
                     batch["energy"].to(device))
        optimizer.zero_grad()
        loss = criterion(model(h, B), tgt)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total += loss.item()
    return total / len(loader)


def evaluate(model, loader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for batch in loader:
            h, B, tgt = (batch["h_feat"].to(device),
                         batch["B_chol"].to(device),
                         batch["energy"].to(device))
            preds.append(model(h, B).cpu())
            targets.append(tgt.cpu())
    preds   = torch.cat(preds).numpy()
    targets = torch.cat(targets).numpy()
    return float(np.mean(np.abs(preds - targets))), preds, targets

In [ ]:
# Cell 7 — 5-Fold Cross-Validation with Real Prediction Tracking
#
def run_cross_validation(data_df, n_folds=5, epochs=1000):
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

    # Out-of-fold arrays aligned with data_df positions (0 .. len-1)
    oof_pred_corr = np.full(len(data_df), np.nan)
    oof_true_corr = np.full(len(data_df), np.nan)
    fold_maes     = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(data_df)):
        print(f"\n--- Fold {fold+1}/{n_folds} ---")

        train_loader = DataLoader(
            MoleculeDataset(data_df.iloc[train_idx]),
            batch_size=8, shuffle=True,  collate_fn=pad_collate)
        val_loader = DataLoader(
            MoleculeDataset(data_df.iloc[val_idx]),
            batch_size=8, shuffle=False, collate_fn=pad_collate)

        model     = FactorizedBipartiteGNN(hidden_dim=64, num_layers=3).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=6e-4)
        criterion = nn.HuberLoss(delta=0.1)
        best_mae, best_state   = float("inf"), None
        patience, max_patience = 0, 150

        for epoch in range(epochs):
            train_epoch(model, train_loader, optimizer, criterion)
            val_mae, _, _ = evaluate(model, val_loader)
            if val_mae < best_mae:
                best_mae, best_state = val_mae, copy.deepcopy(model.state_dict())
                patience = 0
            else:
                patience += 1
            if patience >= max_patience:
                print(f"  Early stop ep {epoch+1}. Best val MAE: {best_mae:.4f} Ha")
                break

        # val_loader shuffle=False -> predictions match val_idx order exactly
        model.load_state_dict(best_state)
        final_mae, preds, targets = evaluate(model, val_loader)

        oof_pred_corr[val_idx] = preds
        oof_true_corr[val_idx] = targets
        fold_maes.append(final_mae)
        print(f"  Fold {fold+1} Final MAE: {final_mae:.4f} Ha")

    print(f"\n[CV Complete] OOF MAE: {np.mean(fold_maes):.4f} +/- {np.std(fold_maes):.4f} Ha")

    df_out = data_df.copy()
    df_out["true_e_corr"] = oof_true_corr
    df_out["pred_e_corr"] = oof_pred_corr
    df_out["pred_E_FCI"]  = df_out["E_HF"] + df_out["pred_e_corr"]
    return fold_maes, df_out


cv_maes, df_with_preds = run_cross_validation(df_main)

In [ ]:
# Cell 8 — Ablation: No Bipartite Message Passing

class OrbitalOnlyGNN(nn.Module):
    # Homogeneous orbital graph: skips the O <-> A message-passing loop entirely.
    # Equivalent to a Deep-Set over orbital node embeddings.
    def __init__(self, node_dim=2, hidden_dim=64):
        super().__init__()
        self.node_embed  = nn.Linear(node_dim, hidden_dim)
        self.energy_head = nn.Sequential(
            nn.Linear(hidden_dim, 128), nn.LeakyReLU(), nn.Dropout(0.1),
            nn.Linear(128, 64),        nn.LeakyReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, h_diag, B_chol):
        # B_chol intentionally unused — this is the ablation
        return self.energy_head(self.node_embed(h_diag).sum(dim=1)).squeeze(-1)


def run_ablation(data_df, n_folds=5, epochs=500):
    print("Running Ablation: No Bipartite Message Passing...\n")
    kf, abl_maes = KFold(n_splits=n_folds, shuffle=True, random_state=42), []

    for fold, (train_idx, val_idx) in enumerate(kf.split(data_df)):
        train_loader = DataLoader(
            MoleculeDataset(data_df.iloc[train_idx]),
            batch_size=8, shuffle=True,  collate_fn=pad_collate)
        val_loader = DataLoader(
            MoleculeDataset(data_df.iloc[val_idx]),
            batch_size=8, shuffle=False, collate_fn=pad_collate)

        model     = OrbitalOnlyGNN(hidden_dim=64).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=6e-4)
        criterion = nn.HuberLoss(delta=0.1)
        best_mae, best_state   = float("inf"), None
        patience, max_patience = 0, 150

        for epoch in range(epochs):
            train_epoch(model, train_loader, optimizer, criterion)
            val_mae, _, _ = evaluate(model, val_loader)
            if val_mae < best_mae:
                best_mae, best_state = val_mae, copy.deepcopy(model.state_dict())
                patience = 0
            else:
                patience += 1
            if patience >= max_patience:
                print(f"  Fold {fold+1}: Early stop ep {epoch+1}. MAE: {best_mae:.4f} Ha")
                break

        model.load_state_dict(best_state)
        abl_maes.append(evaluate(model, val_loader)[0])

    print(f"\n[Ablation] OOF MAE: {np.mean(abl_maes):.4f} +/- {np.std(abl_maes):.4f} Ha")
    return abl_maes


ablation_maes = run_ablation(df_main)

In [ ]:
# Cell 9 — Leave-One-Molecule-Out (LOMO) Cross-Validation

def run_lomo_cv(data_df, epochs=800, val_fraction=0.15):
    molecules    = sorted(data_df["molecule"].unique())
    lomo_results = {}
    print("LOMO-CV (early stopping on training-side val split)...\n")

    for holdout_mol in molecules:
        print(f"--- Holding out: {holdout_mol} ---")
        train_all = data_df[data_df["molecule"] != holdout_mol]
        test_sub  = data_df[data_df["molecule"] == holdout_mol]

        val_sub   = train_all.sample(frac=val_fraction, random_state=42)
        train_sub = train_all.drop(val_sub.index)

        train_loader = DataLoader(
            MoleculeDataset(train_sub), batch_size=8, shuffle=True,  collate_fn=pad_collate)
        val_loader   = DataLoader(
            MoleculeDataset(val_sub),   batch_size=8, shuffle=False, collate_fn=pad_collate)
        test_loader  = DataLoader(
            MoleculeDataset(test_sub),  batch_size=8, shuffle=False, collate_fn=pad_collate)

        model     = FactorizedBipartiteGNN(hidden_dim=64, num_layers=3).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=6e-4)
        criterion = nn.HuberLoss(delta=0.1)
        best_val_mae, best_state   = float("inf"), None
        patience,     max_patience = 0, 150

        for epoch in range(epochs):
            train_epoch(model, train_loader, optimizer, criterion)
            val_mae, _, _ = evaluate(model, val_loader)   
            if val_mae < best_val_mae:
                best_val_mae, best_state = val_mae, copy.deepcopy(model.state_dict())
                patience = 0
            else:
                patience += 1
            if patience >= max_patience:
                print(f"  Early stop ep {epoch+1}. Best val MAE: {best_val_mae:.4f} Ha")
                break

        # One-time evaluation on the genuinely unseen molecule
        model.load_state_dict(best_state)
        test_mae, _, _ = evaluate(model, test_loader)
        print(f"  Zero-shot MAE for {holdout_mol}: {test_mae:.4f} Ha\n")
        lomo_results[holdout_mol] = test_mae

    print("=== LOMO Results ===")
    for mol, mae in sorted(lomo_results.items(), key=lambda x: x[1]):
        print(f"  {mol:5s}: {mae:.4f} Ha")
    return lomo_results


lomo_results = run_lomo_cv(df_main)

In [ ]:
# Cell 10 — Computational Scaling Benchmark
# N_aux = 2*N approximates Beebe-Linderberg linear auxiliary-basis scaling.
# CPU is used for reproducible wall-clock timing (MPS can jitter).

def run_scaling_benchmark(N_values=None, n_repeats=50):
    if N_values is None:
        N_values = [10, 20, 30, 40, 50]
    bench_dev = torch.device("cpu")
    model     = FactorizedBipartiteGNN(hidden_dim=64, num_layers=3).to(bench_dev)
    model.eval()
    times_ms  = []
    print("--- Scaling Benchmark (CPU) ---")
    for N in N_values:
        N_aux = 2 * N
        h = torch.randn(1, N, 2,       device=bench_dev)
        B = torch.randn(1, N_aux, N, N, device=bench_dev)
        for _ in range(5): _ = model(h, B)           # warm-up
        t0 = time.perf_counter()
        with torch.no_grad():
            for _ in range(n_repeats): _ = model(h, B)
        avg_ms = (time.perf_counter() - t0) / n_repeats * 1000
        times_ms.append(avg_ms)
        print(f"  N={N:3d}  N_aux={N_aux:3d}  -> {avg_ms:.2f} ms")
    poly = np.polyfit(np.log(N_values), np.log(times_ms), 1)
    print(f"\nEmpirical scaling exponent: O(N^{poly[0]:.2f})")
    return N_values, times_ms, poly[0]


N_vals, times_ms, scaling_exp = run_scaling_benchmark()

In [ ]:
# Cell 11 — Permutation Invariance Test
#
def test_permutation_invariance(N_test=15):
    model = FactorizedBipartiteGNN(hidden_dim=64, num_layers=3)
    model.eval()
    N_aux  = 2 * N_test
    h_test = torch.randn(1, N_test, 2)

    B_raw  = torch.randn(1, N_aux, N_test, N_test)
    B_test = (B_raw + B_raw.transpose(-1, -2)) / 2.0

    with torch.no_grad():
        E_base = model(h_test, B_test).item()

    perm   = torch.randperm(N_test)
    h_perm = h_test[:, perm, :]
    B_perm = B_test[:, :, perm, :][:, :, :, perm]

    with torch.no_grad():
        E_perm = model(h_perm, B_perm).item()

    diff   = abs(E_base - E_perm)
    status = "PASS" if diff < 1e-5 else "FAIL"
    print(f"Base energy:     {E_base:.8f} Ha")
    print(f"Permuted energy: {E_perm:.8f} Ha")
    print(f"|dE|:            {diff:.2e} Ha  -> {status}")
    return diff


perm_diff = test_permutation_invariance()

In [ ]:
# Cell 12 — Figure 1: Potential Energy Surfaces
#
try:
    plt.style.use("seaborn-v0_8-paper")
except OSError:
    plt.style.use("seaborn-paper")

plt.rcParams.update({
    "font.family": "serif", "font.size": 11,
    "axes.labelsize": 13, "axes.grid": True,
    "grid.alpha": 0.3, "legend.fontsize": 10,
})

mols = sorted(df_main["molecule"].unique())
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharey=False)

for idx, mol in enumerate(mols):
    ax     = axes[idx // 3, idx % 3]
    mol_df = df_with_preds[df_with_preds["molecule"] == mol].sort_values("r")
    mask   = mol_df["pred_E_FCI"].notna()

    # Ground-truth PES
    ax.plot(mol_df["r"], mol_df["E_FCI"], "k-", linewidth=2, label="FCI Exact")

    # Real model predictions (out-of-fold)
    ax.scatter(
        mol_df.loc[mask, "r"], mol_df.loc[mask, "pred_E_FCI"],
        c="#e74c3c", marker="x", s=55, linewidths=1.5,
        label="Bipartite GNN (OOF)", zorder=3,
    )

    ax.set_title(mol)
    ax.set_xlabel(r"Bond Length ($\AA$)")
    if idx % 3 == 0:
        ax.set_ylabel("Energy (Hartree)")
    ax.legend()

plt.suptitle(
    "Figure 1: PES — FCI vs. Bipartite GNN (real out-of-fold predictions)",
    fontsize=12, y=1.01,
)
plt.tight_layout()
plt.savefig("figures/fig1_pes_real_predictions.pdf", dpi=300, bbox_inches="tight")
plt.show()
print("Saved: figures/fig1_pes_real_predictions.pdf")

In [ ]:
# Cell 13 — Figure 2: Computational Scaling

try:
    plt.style.use("seaborn-v0_8-paper")
except OSError:
    plt.style.use("seaborn-paper")

plt.rcParams.update({"font.family": "serif", "font.size": 11, "axes.labelsize": 13})

poly     = np.polyfit(np.log(N_vals), np.log(times_ms), 1)
fit_line = np.exp(np.polyval(poly, np.log(N_vals)))

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(N_vals, times_ms, marker="o", color="#2980b9", linewidth=2,
        label="Bipartite GNN (measured)")
ax.plot(N_vals, fit_line, linestyle="--", color="#c0392b",
        label=fr"Empirical Scaling $\mathcal{{O}}(N^{{{poly[0]:.2f}}})$")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("Number of Orbitals ($N$)")
ax.set_ylabel("Forward Pass Time (ms)")
ax.set_title("Figure 2: Computational Scaling")
ax.legend()
ax.grid(True, which="both", ls="--", alpha=0.3)
plt.tight_layout()
plt.savefig("figures/fig2_computational_scaling.pdf", dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: figures/fig2_computational_scaling.pdf  (exponent={poly[0]:.2f})")

In [ ]:
# Cell 14 — Figure 3: LOMO Latent Anchoring (Delta-Z vs. Zero-Shot MAE)

try:
    plt.style.use("seaborn-v0_8-paper")
except OSError:
    plt.style.use("seaborn-paper")

plt.rcParams.update({"font.family": "serif", "font.size": 11, "axes.labelsize": 13})

z_map    = {"H": 1, "Li": 3, "C": 6, "N": 7, "O": 8, "F": 9}
atom_map = {
    "CO": ("C","O"),   "HF": ("H","F"),   "Li2": ("Li","Li"),
    "LiH":("Li","H"),  "N2": ("N","N"),   "O2": ("O","O"),
}
mol_tex  = {
    "CO":"CO", "HF":"HF", "Li2":r"Li$_2$",
    "LiH":"LiH", "N2":r"N$_2$", "O2":r"O$_2$",
}
text_off = {
    "CO":( 0.3, 0.001), "HF":(-1.2, 0.002),
    "Li2":(0.2,-0.003), "LiH":(0.2, 0.001),
    "N2":(-1.2, 0.001), "O2": (0.3,-0.002),
}

mols_s   = sorted(lomo_results.keys())
dz_vals  = [abs(z_map[atom_map[m][0]] - z_map[atom_map[m][1]]) for m in mols_s]
mae_vals = [lomo_results[m] for m in mols_s]

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(dz_vals, mae_vals, color="#2c3e50", s=100, zorder=3, edgecolors="black")

for mol, dz, mae in zip(mols_s, dz_vals, mae_vals):
    dx, dy = text_off.get(mol, (0.2, 0))
    ax.text(dz + dx, mae + dy, mol_tex[mol], fontsize=12, zorder=4)

z_range = np.linspace(-0.5, 8.5, 200)
fit     = np.polyfit(dz_vals, mae_vals, 1)
ax.plot(z_range, np.polyval(fit, z_range), "--", color="#e74c3c",
        alpha=0.7, label="Linear Trend")

ax.set_xlabel(r"Nuclear Charge Asymmetry $\Delta Z = |Z_A - Z_B|$")
ax.set_ylabel("Zero-Shot MAE (Hartree)")
ax.set_title("Figure 3: LOMO Generalization vs. Nuclear Charge Asymmetry")
ax.set_xlim(-0.5, 8.5)
ax.set_ylim(0, max(mae_vals) * 1.18)
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig("figures/fig3_lomo_latent_anchoring.pdf", dpi=300, bbox_inches="tight")
plt.show()
print("Saved: figures/fig3_lomo_latent_anchoring.pdf")

In [ ]:
# Cell 15 — Results Summary Table

cv_mean,  cv_std  = np.mean(cv_maes),       np.std(cv_maes)
abl_mean, abl_std = np.mean(ablation_maes), np.std(ablation_maes)

SEP = "=" * 62
print(SEP)
print("Table 1: 5-Fold OOF MAE on FCI Correlation Energy Prediction")
print(SEP)
rows = [
    ("Bipartite-Chol (Ours)",   "Factorized Bipartite", f"{cv_mean:.4f} +/- {cv_std:.4f}"),
    ("Ablation: No Aux. Nodes", "Orbital Homogeneous",  f"{abl_mean:.4f} +/- {abl_std:.4f}"),
]
print(f"{'Model':<30} {'Graph Structure':<24} {'MAE (Ha)'}")
print("-" * 62)
for r in rows:
    print(f"{r[0]:<30} {r[1]:<24} {r[2]}")

print()
print(SEP)
print("Table 2: LOMO Zero-Shot MAE by Molecule")
print(SEP)
z_map_t   = {"H":1,"Li":3,"C":6,"N":7,"O":8,"F":9}
atom_map_t = {
    "CO":("C","O"),"HF":("H","F"),"Li2":("Li","Li"),
    "LiH":("Li","H"),"N2":("N","N"),"O2":("O","O"),
}
for mol, mae in sorted(lomo_results.items(), key=lambda x: x[1]):
    dz = abs(z_map_t[atom_map_t[mol][0]] - z_map_t[atom_map_t[mol][1]])
    print(f"  {mol:5s}  dZ={dz}  ->  {mae:.4f} Ha")

print()
print(f"Permutation invariance:     |dE| = {perm_diff:.2e} Ha  "
      f"({'PASS' if perm_diff < 1e-5 else 'FAIL'})")
print(f"Empirical scaling exponent: O(N^{scaling_exp:.2f})")
print(SEP)